# Chain-of-Thought Prompting — Teaching the Model to Think Out Loud

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q boto3 openai tiktoken anthropic matplotlib

## How LLM calls work in this notebook

Every live API call goes through `LLMRouter` from `garage_helper`:

1. **Model resolution** — the router maps your model to the right provider (`anthropic`, `openai`, `azure_openai`, `bedrock_claude`, `gemini`, etc.) automatically.
2. **CoT calls** — Chain-of-Thought is a prompting technique, not an API feature. The router sends the same API call regardless; the only difference is the prompt text includes "Think step by step" or a structured reasoning scaffold.
3. **Token comparison** — `router.generate_response()` returns `.input_tokens` and `.output_tokens` so you can directly measure the token overhead of CoT vs direct answers, across any provider.
4. **Classifier + solver pattern** — the notebook's routing demo makes two sequential `router.generate()` calls: one short call to classify the problem, then one to solve it. The provider instance is cached so the client is created once and reused.

These notebooks have been tested with **Claude** (via Anthropic direct API and AWS Bedrock) and **GPT** models (via Azure OpenAI and direct OpenAI). Set `verbose=True` to watch both calls logged in cell output.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from garage_helper import setup_llm, LLMRouter

# Contributors: add DEFAULT_LLM_MODEL (and any provider credentials) to a .env at the repo root.
# Learners: setup_llm() will run an interactive wizard to pick a provider and enter credentials.
MODEL  = setup_llm()
router = LLMRouter(default_model=MODEL, verbose=False)

## When better prompts stop helping

In the previous notebooks — Zero-Shot, Few-Shot, and Controlling Output — we covered the mechanics of shaping what a model produces. You can specify format, register, length, schema. You can demonstrate patterns with examples. For a surprisingly large class of tasks, that is enough.

But then you hit a wall. The model gets the format exactly right and still produces a wrong answer. You tighten the system prompt. Still wrong. You add examples. It pattern-matches to the examples and fails on any problem that looks slightly different. The issue is not the output shape — it is the reasoning path.

This happens reliably on multi-step problems: arithmetic that requires carrying intermediate results, logic puzzles that require eliminating options, scheduling problems that require propagating constraints forward. For these tasks, the model's default behaviour — compress the question and emit an answer directly — skips the steps where errors compound. By the time it writes the final number, it has already committed to the wrong path.

Chain-of-Thought (CoT) prompting is the fix. Instead of asking for the answer, you ask the model to work through the problem step by step — out loud, in the context window — before committing to a conclusion. The scratchpad becomes part of the output. The reasoning becomes visible and correctable. And the accuracy on hard problems improves dramatically.

## Concept 1 — Zero-shot CoT: "Think step by step"

The simplest version of CoT costs you exactly four words. Add "Think step by step" to your prompt and the model externalises its reasoning before answering.

**The analogy:** imagine a student taking a maths exam. Normally they circle the answer and move on. Tell them to "show all working" and two things happen: their accuracy on hard problems goes up, and you can spot exactly where they went wrong on the ones they still miss. The model's reasoning window is the exam paper. "Think step by step" is the instruction to show working.

This was formalised in the paper *Large Language Models are Zero-Shot Reasoners* (Kojima et al., 2022), which showed that appending a single sentence was enough to dramatically improve performance on reasoning benchmarks — no examples required. The mechanism is straightforward: generating intermediate tokens forces the model's next-token predictions to be conditioned on explicit partial results rather than a compressed internal representation of the whole problem.

In [ ]:
problem = (
    "A shop buys 40 notebooks at £1.20 each and 25 pens at £0.80 each. "
    "They sell all the notebooks at £2.50 each and all the pens at £1.50 each. "
    "What is the total profit?"
)

system = "You are a precise problem solver. Answer concisely."

direct_resp = router.generate_response(problem, model=MODEL, system=system, max_tokens=120)
cot_resp    = router.generate_response(
    problem + " Think step by step.", model=MODEL, system=system, max_tokens=400
)

print("DIRECT (no reasoning instruction):")
print(direct_resp.text.strip())
print(f"  tokens used: {direct_resp.output_tokens}")

print()
print("ZERO-SHOT CoT ('Think step by step'):")
print(cot_resp.text.strip())
print(f"  tokens used: {cot_resp.output_tokens}")

print()
print("Correct answer: cost = £68, revenue = £137.50, profit = £69.50")
print("CoT uses more tokens but the working is auditable — you can see exactly where each number came from.")

## Concept 2 — Few-shot CoT: worked examples as reasoning templates

Zero-shot CoT is fast and surprisingly effective, but it leaves the structure of the reasoning up to the model. For tasks where you care not just about the final answer but about a specific *format* of reasoning — extracting premises, labelling inference steps, outputting confidence — you need worked examples.

**The analogy:** think about learning long division at school. The teacher does not just say "figure it out step by step." They write three problems on the board with each step labelled: divide, multiply, subtract, bring down. You see the scaffold before you use it. Few-shot CoT is that chalkboard. The examples are not there to demonstrate what the answer looks like — they are there to demonstrate what the *reasoning path* looks like.

The critical difference from regular few-shot prompting: in regular few-shot you show `input → output`. In few-shot CoT you show `input → reasoning chain → output`. The chain is the point. The model learns to generate intermediate steps that match the style and granularity of your examples before reaching a conclusion.

In [ ]:
system = "You are a maths tutor. Always show your reasoning before giving the final classification."

# Build few-shot CoT as a flattened history prompt
few_shot_history = (
    "User: Sam has 12 apples and gives 5 to a friend. How many does Sam have?\n"
    "Assistant: Step 1 — identify what changes: Sam starts with a quantity and loses some.\n"
    "Step 2 — losing a quantity from a total means we remove it.\n"
    "Step 3 — removing is subtraction.\n"
    "Classification: SUBTRACTION\n\n"
    "User: A box holds 8 cans. There are 6 boxes. How many cans are there in total?\n"
    "Assistant: Step 1 — identify the structure: a repeated quantity (8 cans) across multiple groups (6 boxes).\n"
    "Step 2 — repeated addition of equal groups is multiplication.\n"
    "Step 3 — 8 × 6 gives the total.\n"
    "Classification: MULTIPLICATION\n\n"
    "User: A runner completes a 42 km marathon. "
    "The race is split into 6 equal segments for water stations. "
    "How long is each segment?"
)

response = router.generate(few_shot_history, model=MODEL, system=system, max_tokens=200)

print("Few-shot CoT — the model follows the reasoning scaffold from the examples:")
print()
print(response.strip())
print()
print("The model adopted the 'Step N —' format and the closing Classification label.")
print("The examples taught the reasoning structure, not just the answer format.")

## Concept 3 — Why CoT works: the scratchpad changes the math

Understanding *why* CoT works makes it easier to know when to reach for it — and when not to.

A Transformer generates one token at a time. Each token is predicted given everything that came before it in the context window. When a model answers a multi-step problem directly, it has to compress the entire chain of reasoning into its hidden state and emit the final answer in one shot. The hidden state has finite capacity, and for problems with many interdependent steps, that capacity runs out — information from early steps gets overwritten before it influences late steps.

**The analogy:** try multiplying 347 × 29 in your head with no pen. You probably lose track of a partial product somewhere. Now try it with paper. The intermediate numbers you write down are not extra work — they are external memory that extends the effective working space of your brain. The model's context window is the paper. CoT tokens are the intermediate numbers written down.

When the model generates "Step 1: cost of notebooks = 40 × £1.20 = £48", that result is now a token in the context window. Every subsequent token is predicted with £48 explicitly present as a conditioning signal. The model doesn't have to recompute or remember it. This is the mechanical reason CoT helps: it converts an opaque compression task into a sequential reading task, where each step builds on verified prior steps.

In [ ]:
puzzle = """
Five people — Alice, Bob, Carlos, Diana, and Eve — sit in a row of five seats numbered 1 to 5.
Constraints:
  1. Alice is not in seat 1 or seat 5.
  2. Bob sits immediately to the left of Carlos.
  3. Diana sits in an odd-numbered seat.
  4. Eve sits in seat 1 or seat 5.
  5. Alice and Diana are not adjacent.

Who sits in seat 3?
""".strip()

system = "You are a logical puzzle solver."

direct = router.generate_response(
    puzzle + "\nAnswer with just the name.", model=MODEL, system=system, max_tokens=60
)
cot = router.generate_response(
    puzzle + "\nWork through each constraint in order, eliminating possibilities, then state who is in seat 3.",
    model=MODEL, system=system, max_tokens=600
)

print("DIRECT answer:")
print(f"  {direct.text.strip()}")
print(f"  ({direct.output_tokens} output tokens)")

print()
print("CHAIN-OF-THOUGHT reasoning:")
print(cot.text.strip())
print(f"  ({cot.output_tokens} output tokens)")

print()
print("CoT trades tokens for accuracy. Each constraint elimination is a checkpoint —")
print("if the final answer is wrong, you can see exactly which step went off the rails.")

## Concept 4 — When CoT helps and when it wastes tokens

CoT is not free. Every reasoning step is tokens in and tokens out. For a simple question, those tokens add latency, cost, and noise without improving the answer. Knowing when to use CoT is as important as knowing how.

**The analogy:** a surgeon washing hands before a routine check-up is overkill — it takes two minutes and adds nothing. The same surgeon not scrubbing before a four-hour open-heart procedure is catastrophic. The procedure determines the preparation. Chain-of-thought is the scrub — skip it for the check-up, never skip it for the surgery.

The deciding factor is whether the answer requires **chaining dependent intermediate results**. If yes — multi-step arithmetic, logic with several constraints, scheduling with conflicts, code that requires tracking state across multiple operations — CoT genuinely helps. If no — simple factual recall, single-step classification, direct format conversion, lookup-style questions — CoT burns tokens for zero gain and can actually introduce errors by giving the model more surface area to hallucinate on.

In [ ]:
system = "You are a helpful assistant. Be precise."

tasks = [
    {
        "label":   "Simple factual lookup",
        "direct":  "What is the capital of Japan?",
        "cot":     "What is the capital of Japan? Think step by step.",
        "verdict": "CoT wastes tokens — no intermediate steps exist",
    },
    {
        "label":   "Single-step classification",
        "direct":  "Is the word 'serendipity' a noun, verb, or adjective? Answer in one word.",
        "cot":     "Is the word 'serendipity' a noun, verb, or adjective? Think step by step, then give one word.",
        "verdict": "CoT adds preamble but the answer doesn't change",
    },
    {
        "label":   "Multi-step rate problem",
        "direct":  "A tank fills at 8 litres/min and drains at 3 litres/min. Starting empty, how full is it after 12 minutes?",
        "cot":     "A tank fills at 8 litres/min and drains at 3 litres/min. Starting empty, how full is it after 12 minutes? Think step by step.",
        "verdict": "CoT helps — net rate must be computed before applying time",
    },
    {
        "label":   "Chained conditional logic",
        "direct":  "If it rains, the match is cancelled. If the match is cancelled, the team dinner is moved to Friday. If the team dinner is on Friday, Ana can't attend because she flies out Thursday night. It rained. Can Ana attend the dinner?",
        "cot":     "If it rains, the match is cancelled. If the match is cancelled, the team dinner is moved to Friday. If the team dinner is on Friday, Ana can't attend because she flies out Thursday night. It rained. Can Ana attend the dinner? Think step by step.",
        "verdict": "CoT helps — three conditional hops must be traced in order",
    },
]

print(f"{'Task':<30}  {'Direct tokens':>13}  {'CoT tokens':>10}  {'Token delta':>11}  Verdict")
print("-" * 100)

for task in tasks:
    direct_r = router.generate_response(task["direct"], model=MODEL, system=system, max_tokens=200)
    cot_r    = router.generate_response(task["cot"],    model=MODEL, system=system, max_tokens=400)
    d_tok = direct_r.output_tokens
    c_tok = cot_r.output_tokens
    delta = c_tok - d_tok
    print(f"{task['label']:<30}  {d_tok:>13}  {c_tok:>10}  {'+' + str(delta):>11}  {task['verdict']}")

print()
print("For simple tasks: CoT adds tokens with no accuracy gain.")
print("For chained multi-step tasks: the extra tokens are doing real work.")

## Concept 5 — Structuring the chain: controlling the reasoning format

"Think step by step" works, but it leaves the granularity and structure of the chain up to the model. For production use — where reasoning steps might be logged, audited, or displayed to users — you often want a more controlled shape.

**The analogy:** a doctor's diagnostic notes follow a standard format — symptoms, differentials, investigations, conclusion — for a reason. Anyone reading the notes can follow the logic. An unstructured paragraph saying "I thought about various options and concluded X" is useless to a second doctor reviewing the case. Structured CoT is the SOAP note: each section has a name, a purpose, and an expected content.

You can ask for almost any reasoning format by describing it in the prompt or demonstrating it in an example. Common useful structures: numbered steps, `<thinking>` / `<answer>` XML blocks (used by models like Claude natively), bullet-per-constraint for logic puzzles, or an explicit scratchpad/conclusion split.

In [ ]:
problem = """
We need to schedule a 1-hour team meeting this week. Here are the constraints:
- Alice is unavailable Monday all day and Wednesday afternoon (after 2pm).
- Bob is in external meetings Tuesday and Thursday morning (before 12pm).
- Carlos is on leave Friday.
- The meeting must be between 9am–5pm on a weekday.
- No meeting on a day where two or more people are entirely unavailable.

What is the earliest possible 1-hour slot this week?
""".strip()

formats = [
    (
        "Numbered steps",
        problem + "\n\nWork through each day in order as numbered steps. End with: ANSWER: <slot>"
    ),
    (
        "Scratchpad + conclusion split",
        problem + "\n\nFirst write a SCRATCHPAD section where you eliminate options. Then write a CONCLUSION section with the final answer only."
    ),
    (
        "Constraint table",
        problem + "\n\nFirst produce a table of each person's availability per day, then identify the earliest open slot."
    ),
]

system = "You are a precise scheduling assistant."

for label, prompt in formats:
    response = router.generate(prompt, model=MODEL, system=system, max_tokens=500)
    print(f"[Format: {label}]")
    print(response.strip())
    print("-" * 60)
    print()

print("Same problem, three reasoning structures.")
print("The format you choose should match how downstream consumers will read or parse the output.")

## Putting it together — a CoT decision framework in code

Let's wire everything up: a small dispatcher that classifies an incoming problem as direct-answer or chain-of-thought territory, then routes it accordingly and surfaces the reasoning only when it exists.

In [ ]:
CLASSIFIER_SYSTEM = """
You classify questions as needing chain-of-thought reasoning or not.
Reply with exactly one word: COT or DIRECT.
COT = multi-step maths, logic chains, scheduling with conflicts, code tracing with state.
DIRECT = factual lookup, single-step classification, simple format conversion.
""".strip()

SOLVER_SYSTEM = "You are a precise problem solver."

def solve(question: str) -> dict:
    classification = router.generate(
        question, model=MODEL, system=CLASSIFIER_SYSTEM, max_tokens=5
    ).strip().upper()

    use_cot = classification == "COT"

    if use_cot:
        prompt  = question + "\n\nWork through this step by step. Label each step. End with 'Final answer: <answer>'"
        max_tok = 500
    else:
        prompt  = question
        max_tok = 80

    resp = router.generate_response(prompt, model=MODEL, system=SOLVER_SYSTEM, max_tokens=max_tok)

    return {
        "question": question[:70] + ("..." if len(question) > 70 else ""),
        "mode":     "CoT" if use_cot else "Direct",
        "tokens":   resp.output_tokens,
        "answer":   resp.text.strip(),
    }

questions = [
    "What programming language was Python named after?",
    "A train leaves London at 09:15 travelling at 90 mph. Another leaves Birmingham (115 miles away) at 09:45 travelling at 70 mph toward London. When do they meet?",
    "Convert the word 'running' to its base form.",
    "Three friends split a £74.40 restaurant bill. One had a voucher covering 15% of the total. How much does each person pay after the voucher is applied?",
]

for q in questions:
    result = solve(q)
    print(f"[{result['mode']}, {result['tokens']} tokens]")
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print()

print("The classifier adds a small overhead but prevents unnecessary CoT on trivial questions.")
print("In production: cache classifications for recurring question patterns.")

## Visualising the CoT accuracy advantage

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Illustrative accuracy and token cost data based on published CoT benchmarks
# (Wei et al. 2022, Kojima et al. 2022 — scaled to be representative, not exact)

task_categories = [
    "Factual\nlookup",
    "Single-step\nclassification",
    "2-step\narithmetic",
    "Multi-step\nword problem",
    "Logic\npuzzle",
    "Constraint\nscheduling",
]

direct_accuracy = [0.95, 0.92, 0.85, 0.58, 0.42, 0.38]
cot_accuracy    = [0.94, 0.91, 0.88, 0.87, 0.83, 0.79]

# Relative token cost multiplier for CoT vs direct
token_cost_ratio = [1.0, 1.1, 1.8, 3.2, 4.5, 5.1]

x = np.arange(len(task_categories))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: accuracy comparison
b1 = ax1.bar(x - width/2, direct_accuracy, width, label="Direct answer",
             color="steelblue", alpha=0.88)
b2 = ax1.bar(x + width/2, cot_accuracy,    width, label="Chain-of-Thought",
             color="seagreen", alpha=0.88)

ax1.set_ylabel("Accuracy (illustrative)")
ax1.set_title("Accuracy: Direct vs Chain-of-Thought")
ax1.set_xticks(x)
ax1.set_xticklabels(task_categories, fontsize=9)
ax1.set_ylim(0, 1.1)
ax1.axhline(0.8, color="grey", linewidth=0.7, linestyle="--", alpha=0.5)
ax1.text(len(x) - 0.3, 0.82, "80%", fontsize=8, color="grey")
ax1.legend(fontsize=9)

# Right: token cost multiplier
colours = ["tomato" if r > 1.5 else "steelblue" for r in token_cost_ratio]
ax2.bar(x, token_cost_ratio, color=colours, alpha=0.85)
ax2.axhline(1.0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax2.set_ylabel("Token cost multiplier vs direct")
ax2.set_title("Token cost: CoT overhead by task type")
ax2.set_xticks(x)
ax2.set_xticklabels(task_categories, fontsize=9)
ax2.text(-0.5, 1.08, "1× = same cost as direct", fontsize=8, color="grey")

cheap_patch  = mpatches.Patch(color="steelblue", label="Low CoT overhead (< 1.5×)")
costly_patch = mpatches.Patch(color="tomato",    label="High CoT overhead (≥ 1.5×)")
ax2.legend(handles=[cheap_patch, costly_patch], fontsize=9)

plt.tight_layout()
plt.show()

print("Left chart: CoT barely moves accuracy on simple tasks — it matches or ties direct.")
print("Left chart: CoT lifts accuracy significantly on multi-step and logic tasks.")
print("Right chart: the token cost is low for simple tasks and high for complex ones.")
print("Rule of thumb — if CoT won't change the answer, skip it. The cost is real.")

## Key takeaways

- **Chain-of-Thought works by externalising intermediate steps** into the context window, where they act as explicit conditioning for subsequent tokens — the same mechanism as writing intermediate results on paper instead of keeping them in your head.
- **Zero-shot CoT** ("Think step by step") is the fastest entry point: four words that reliably improve accuracy on multi-step problems with no examples required.
- **Few-shot CoT** shows the model a reasoning *structure*, not just a format. Include the chain in your example outputs, not just the final label — that is what makes it different from regular few-shot.
- **CoT is worth the token cost on chained problems** — multi-step arithmetic, logic with several constraints, scheduling conflicts, code tracing. It is not worth it on direct-recall or single-step tasks.
- **Reasoning format is controllable**: numbered steps, scratchpad/conclusion splits, XML blocks, constraint tables — describe or demonstrate the structure you need and the model will follow it.
- **CoT makes failures auditable**: if the final answer is wrong, you can read the chain and see exactly which step went off the rails. Silent wrong answers are harder to debug than wrong-step visible reasoning.
- **A simple classifier can route requests** automatically between direct and CoT modes, keeping simple queries cheap while routing hard ones through the full reasoning chain.

---

Next up: **Controlling the Output** — now that the model is reasoning out loud, let's look at how to govern what it actually produces: format, length, tone, structured JSON, and the tools that shape generation at the API level.